# 08 — Verified Bilingual Parent Reports (`backend/app/genai`)

**Agent E Deliverable** — Zero-hallucination, deterministic-verifier-gated English (`en`) and Hindi (`hi`) weekly parent reports.

### Architecture Overview
1. **Numbers-First Input (`NumbersJson`)**: Every quantitative claim (`attendance_pct`, `tests_taken`, `trend`, `strengths`, `focus`, `home_tips`) is computed deterministically upstream before any language model or template runs.
2. **Strict 8-Stage Verifier (`verify(output, numbers, lang)`)**: Validates JSON schema (`contracts/report_output.schema.json`), number/Devanagari-digit/number-word allow-list, 380-topic NTA syllabus allow-list (`contracts/topics.yaml`), proper-name & placeholder hygiene, banned fear/shame/forecasting lexicon (`contracts/reason_templates.yaml`), Unicode script & zero-width security, word/sentence length bounds (80–180 words, ≤22 words/sentence), and trend-direction alignment.
3. **Guaranteed Fallback (`render_template_output` / `NullClient`)**: If the LLM times out, produces invalid JSON, fails any verifier check after 1 self-correction retry, or exceeds the institute's monthly budget cap, `generate_report()` falls back to 10 English + 10 Hindi human-authored Jinja2 templates that pass the exact same `verify()` gate.
4. **Post-Verification Placeholder Substitution**: `{STUDENT}` and `{INSTITUTE}` are substituted *only after* `verify()` succeeds (`substitute_placeholders`), preventing student/institute PII from entering the LLM context or prompt cache.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from backend.app.genai import (
    CostMeter,
    FakeClient,
    NullClient,
    build_numbers,
    generate_report,
    substitute_placeholders,
    verify,
)
from backend.app.genai.bench import (
    build_50_varied_numbers,
    check_ollama_models_available,
    run_1000_fake_client_mix,
    run_fuzz_500,
    run_ollama_benchmark,
)
from backend.app.genai.pipeline import export_report_pdf_bytes

samples = build_50_varied_numbers()
sample_en, sample_hi = samples[4], samples[1]
print(f'Loaded {len(samples)} synthetic NumbersJson payloads.')
print(f'Sample EN: period={sample_en.period}, attendance={sample_en.attendance_pct}%, tests={sample_en.tests_taken}, trend={sample_en.trend}')
print(f'Sample HI: period={sample_hi.period}, attendance={sample_hi.attendance_pct}%, tests={sample_hi.tests_taken}, trend={sample_hi.trend}')

Loaded 50 synthetic NumbersJson payloads.
Sample EN: period=2026-W05, attendance=91.0%, tests=8, trend=improving
Sample HI: period=2026-W02, attendance=100.0%, tests=1, trend=improving


## 1. Sample Verified Reports (English & Hindi) + Devanagari PDF Smoke Check

Below we generate verified reports using `NullClient` (zero-LLM deterministic mode), inspect the verifier audit, perform post-verification `{STUDENT}` / `{INSTITUTE}` substitution, and verify embedded `NotoSansDevanagari-Regular.ttf` PDF rendering.

In [1]:
rep_en = generate_report(sample_en, lang='en', client=NullClient())
rep_hi = generate_report(sample_hi, lang='hi', client=NullClient())
ver_en = verify(rep_en.content, sample_en, 'en')
ver_hi = verify(rep_hi.content, sample_hi, 'hi')

sub_en = substitute_placeholders(rep_en, student_name='Aarav Verma', institute_name='Sankalp Science Academy')
sub_hi = substitute_placeholders(rep_hi, student_name='आरव वर्मा', institute_name='संकल्प साइंस एकेडमी')

pdf_en = export_report_pdf_bytes(sub_en, lang='en')
pdf_hi = export_report_pdf_bytes(sub_hi, lang='hi')

print(f'=== English Verified Report (path={rep_en.path}, template={rep_en.template_id}, verified={ver_en.ok}, latency={rep_en.latency_ms:.2f} ms) ===')
print(f"Title: {sub_en['title']}\nBody ({sub_en['word_count']} words):\n  {sub_en['body']}")
print(f'PDF Smoke Export: {len(pdf_en)} bytes (valid PDF header={pdf_en[:4] == b"%PDF"})\n')

print(f'=== Hindi Verified Report (path={rep_hi.path}, template={rep_hi.template_id}, verified={ver_hi.ok}, latency={rep_hi.latency_ms:.2f} ms) ===')
print(f"Title: {sub_hi['title']}\nBody ({sub_hi['word_count']} words):\n  {sub_hi['body']}")
print(f'PDF Smoke Export: {len(pdf_hi)} bytes (valid PDF header={pdf_hi[:4] == b"%PDF"})')

=== English Verified Report (path=template, template=en_01_improving_balanced, verified=True, latency=116.74 ms) ===
Title: Progress Update (2026-W05)
Body (108 words):
  Dear Parent, this update covers period 2026-W05 for Aarav Verma at Sankalp Science Academy. During this period, Aarav Verma recorded 91% class attendance and completed 8 practice tests. Overall academic progress shows an improving trend across recent assessments. Current strong chapters include Relative Velocity in 1D and 2D, where concept application has been steady. Faculty mentors will continue guiding regular chapter practice across the syllabus. At home, you can support Aarav Verma with two simple habits. First, solve one short timed practice set at home without interruption. Second, ensure adequate nightly sleep and a calm weekly rest day. Our faculty team at Sankalp Science Academy is happy to support Aarav Verma in every weekly review session.
PDF Smoke Export: 15551 bytes (valid PDF header=True)

=== Hindi Ve

## 2. 500-Case Adversarial Verifier Fuzz Suite (`12` Corruption Categories)

Target bar from `contracts/metrics_bars.yaml`: `genai_verifier_injected_error_catch_rate == 1.0` (`100%` catch rate across all 500 injected corruptions).

In [1]:
fuzz = run_fuzz_500()
print(f"Total Corrupted Payloads Tested: {fuzz['total_corrupted']}")
print(f"Total Caught by Verifier:        {fuzz['caught']}")
print(f"genai_verifier_injected_error_catch_rate: {fuzz['catch_rate']:.4f} (Target Bar: == 1.0000)\n")
print(f"{'Corruption Category':<34} | {'Tested':>6} | {'Caught':>6} | {'Catch Rate':>10}")
print('-' * 66)
for cat, stats in fuzz['by_category'].items():
    print(f"{cat:<34} | {stats['total']:>6} | {stats['caught']:>6} | {stats['catch_rate']:>9.1%}")

Total Corrupted Payloads Tested: 500
Total Caught by Verifier:        500
genai_verifier_injected_error_catch_rate: 1.0000 (Target Bar: == 1.0000)

Corruption Category                | Tested | Caught | Catch Rate
------------------------------------------------------------------
wrong_number_ascii                 |     42 |     42 |    100.0%
wrong_number_devanagari_or_word    |     42 |     42 |    100.0%
unallowed_topic_swap               |     42 |     42 |    100.0%
duplicate_focus_topic              |     42 |     42 |    100.0%
rank_or_comparison                 |     42 |     42 |    100.0%
proper_name_injection              |     42 |     42 |    100.0%
dropout_or_mental_health_wording   |     42 |     42 |    100.0%
missing_placeholder                |     42 |     42 |    100.0%
length_or_word_count_violation     |     41 |     41 |    100.0%
inverted_trend                     |     41 |     41 |    100.0%
unicode_confusable_or_zero_width   |     41 |     41 |    100.0%
mixe

## 3. 1,000-Report Chaos & Fallback Simulation (`FakeClient` Mix)

Simulates 1,000 weekly report generations where `FakeClient` returns mixtures of valid JSON, retry-recovered outputs, and double-corrupted outputs. Verifies that `genai_unverified_reports_shipped == 0`.

In [1]:
mix = run_1000_fake_client_mix()
null_lats = sorted([generate_report(samples[i % len(samples)], client=NullClient()).latency_ms for i in range(100)])
print('=== 1,000-Report Chaos & Fallback Simulation ===')
print(f"Total Reports Generated:           {mix['total_generated']}")
print(f"Served via Verified LLM Path:      {mix['llm_accepted']}")
print(f"Served via Deterministic Fallback: {mix['template_fallbacks']}")
print(f"Unverified Reports Shipped:        {mix['unverified_shipped']} (Target Bar: == 0)")
print(f"NullClient Zero-LLM Latency (100 runs): p50={null_lats[50]:.2f} ms, p95={null_lats[95]:.2f} ms, max={max(null_lats):.2f} ms")

=== 1,000-Report Chaos & Fallback Simulation ===
Total Reports Generated:           1000
Served via Verified LLM Path:      692
Served via Deterministic Fallback: 308
Unverified Reports Shipped:        0 (Target Bar: == 0)
NullClient Zero-LLM Latency (100 runs): p50=4.44 ms, p95=6.05 ms, max=6.64 ms


## 4. Unit Economics (`CostMeter` & `pricing.yaml`)

Formula:
$$\text{cost}_{\text{INR}} = \frac{\text{in\_tok} \cdot p_{\text{in}} + \text{out\_tok} \cdot p_{\text{out}}}{10^6}$$

Local Ollama models (`qwen3:4b`, `gemma3:4b`) have `₹0.00` marginal API cost. Hosted reference models are included with `verified: false` and explicit source notes.

In [1]:
meter = CostMeter(monthly_cap_inr=500.0)
print('=== Unit Economics: Per-Report & Per-Student/Month (4 reports/mo, in_tok=600, out_tok=300) ===')
print(f"{'Model ID':<26} | {'Provider':<14} | {'INR/Report':>12} | {'INR/Student/Mo':>16} | {'Verified':<8}")
print('-' * 86)
for model_id, info in meter.models.items():
    est = meter.estimate_monthly_student_cost_inr(model_id, avg_tokens_in=600, avg_tokens_out=300, reports_per_month=4)
    print(
        f"{model_id:<26} | {str(info.get('provider')):<14} | {est['cost_per_report_inr']:>12.4f} | "
        f"{est['cost_per_student_month_inr']:>16.4f} | {str(info.get('verified')):<8}"
    )

=== Unit Economics: Per-Report & Per-Student/Month (4 reports/mo, in_tok=600, out_tok=300) ===
Model ID                   | Provider       |   INR/Report |   INR/Student/Mo | Verified
--------------------------------------------------------------------------------------
qwen3:4b                   | ollama_local   |       0.0000 |           0.0000 | True    
gemma3:4b                  | ollama_local   |       0.0000 |           0.0000 | True    
template_fallback          | deterministic_template |       0.0000 |           0.0000 | True    
fake_client                | test_double    |       0.0000 |           0.0000 | True    
gemini-2.5-flash-lite      | google_cloud_vertex_reference |       0.0162 |           0.0648 | False   
gemini-2.5-flash           | google_cloud_vertex_reference |       0.0837 |           0.3348 | False   
gemini-2.5-pro             | google_cloud_vertex_reference |       0.3375 |           1.3500 | False   


## 5. Optional Local LLM Benchmark (`qwen3:4b` vs `gemma3:4b`)

Checks whether `qwen3:4b` and `gemma3:4b` have been pulled into the local Ollama daemon. If present, runs the 50-fixture × 2-language benchmark (100 reports per model); otherwise exits cleanly.

In [1]:
avail = check_ollama_models_available()
print(f'Ollama local model availability check: {avail}')
if all(avail.values()):
    bench_res = run_ollama_benchmark()
    print(bench_res)
else:
    print('Skipping live Ollama benchmark because qwen3:4b and/or gemma3:4b are not yet pulled.')
    print('Ask the human to run:\n  ollama pull qwen3:4b\n  ollama pull gemma3:4b')

Ollama local model availability check: {'qwen3:4b': False, 'gemma3:4b': False}
Skipping live Ollama benchmark because qwen3:4b and/or gemma3:4b are not yet pulled.
Ask the human to run:
  ollama pull qwen3:4b
  ollama pull gemma3:4b
